# logit-thermo A2-13：塩の効き方は大きさで変わるか（Pythia-70M・160M・410M・1B、SmolLM2-135M・360M、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは約 1.5〜2 時間。終わったら自動で切れます。
- ゲート：`docs/A2_13_GATE.md`。腕は「なし」「なし（別の種）」「偽薬 30%」「偽薬 30%（別の種）」の 4 本、温度は 0.4〜0.7 の 7 個（0.05 刻み）。3 本のノートブック（small・mid・large）を 3 枠で並べて回せます。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_13_collect.py`（`fb19164812078c8b…`）は埋め込み。
- 保存先 `A2_13/`。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。
- **通知**：🚀・腕ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-13 塩の効きと大きさ データ集め（small：Pythia-70M・160M・410M・1B、SmolLM2-135M・360M、偽薬 30%、温度 0.4〜0.7、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-13 塩の効きと大きさ データ集め（small：Pythia-70M・160M・410M・1B、SmolLM2-135M・360M、偽薬 30%、温度 0.4〜0.7、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-13 salt vs size small', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0'})
SRC22, SRC24, SRC7C, SRC7BC = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC']
SRC13C = '"""A2-13（ゲート docs/A2_13_GATE.md）のデータ集め：塩（偽薬 30%）の効き方は、Pythia の大きさで変わるか・SmolLM2 でも同じか。GPU（L4）。\nscripts/a2_2_collect.py・scripts/a2_4_collect.py・scripts/a2_7_collect.py・scripts/a2_7b_collect.py を実行したあとの同じ名前空間で実行する。\ngenerate_7b（偽薬の禁止のしかた）は A2-7b・A2-10 のまま使う。温度の並び・腕・種は C13_TEMPS・C13_ARMS・C13_SEED として別の名前で持ち、\nA2-7b の B7_TEMPS・B7_ARMS・SEED7B は置きかえない（2026-10-30 18:49 の Colab で、置きかえたせいで A2-7b の自己テストが止まったため）。人の続きの ℓ（teacher_7b）は出さない\n（T_0 は使わないため。2.8B で人の続きをまとめて通すとメモリが足りないため）。2.8B だけ、20 本ずつ 2 回に分けて生成する（A2-5 と同じ）。\nA → L の回数が少なく区間が広い（A2-10 の Pythia-1B で ±0.25）ので、「なし」と「偽薬 30%」をそれぞれ別の種で 2 本ずつ（合わせてサンプル 4）にする。\n種は gen_seed13・coin_seed13（名前が _b で終わる腕だけ ＋500000）。A2-7b の gen_seed・coin_seed は使わない。\n"""\nimport gc, os, time\nimport numpy as np\nimport torch\n\nC13_TEMPS = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]\nC13_ARMS = [(\'none\', \'none\', 0.0), (\'none_b\', \'none\', 0.0), (\'pl300\', \'placebo\', 0.30), (\'pl300_b\', \'placebo\', 0.30)]\nC13_SEED = 20261123\nC13_GROUPS = {\n    \'small\': [(\'pythia-70m\', \'EleutherAI/pythia-70m\', None), (\'pythia-160m\', \'EleutherAI/pythia-160m\', None),\n              (\'pythia-410m\', \'EleutherAI/pythia-410m\', None), (\'smollm2-135m\', \'HuggingFaceTB/SmolLM2-135M\', None),\n              (\'smollm2-360m\', \'HuggingFaceTB/SmolLM2-360M\', None), (\'pythia-1b\', \'EleutherAI/pythia-1b\', None)],\n    \'mid\': [(\'pythia-1.4b\', \'EleutherAI/pythia-1.4b\', None), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\', None)],\n    \'large\': [(\'pythia-2.8b\', \'EleutherAI/pythia-2.8b\', 20)],\n}\n\n\ndef gen_seed13(k, s, arm):\n    return C13_SEED + 1000 * k + s + (500000 if arm.endswith(\'_b\') else 0)\n\n\ndef coin_seed13(k, s, arm):\n    return C13_SEED + 900000 + 1000 * k + s + (500000 if arm.endswith(\'_b\') else 0)\n\n\ndef gen_chunked(model, prompts, T, valid, gen, H2, kind, dose, rng, chunk):\n    """chunk 本ずつ generate_7b を呼んでつなぐ（同じ gen・rng を続けて使う。決まった順なので結果は決まる）。"""\n    if not chunk or chunk >= len(prompts):\n        return generate_7b(model, prompts, T, valid, gen, H2, kind, dose, rng)\n    parts = [generate_7b(model, prompts[i:i + chunk], T, valid, gen, H2, kind, dose, rng) for i in range(0, len(prompts), chunk)]\n    return {k: np.concatenate([p[k] for p in parts], 0) for k in parts[0]}\n\n\ndef run_13(name, hf_id, chunk, docs, outdir, device, notify=print):\n    todo = [a for a in C13_ARMS if not os.path.exists(f\'{outdir}/{name}@{a[0]}.npz\')]\n    if not todo:\n        print(name, \'はもう全部終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    H2 = h2_of(len(tok))\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nD = len(use)\n    assert len({len(pairs[i][0]) for i in use}) == 1\n    prompts = [pairs[i][0] for i in use]\n    keys_f, keys_b = [\'ell\', \'ban_mass\'], [\'cand\', \'banned\', \'banned_top1\']\n    for arm, kind, dose in todo:\n        ta = time.time()\n        G = {k: np.zeros((len(C13_TEMPS), SAMPLES, nD, CONT), np.float32) for k in keys_f}\n        G.update({k: np.zeros((len(C13_TEMPS), SAMPLES, nD, CONT), np.bool_) for k in keys_b})\n        toks = np.zeros((len(C13_TEMPS), SAMPLES, nD, CONT), np.int32)\n        for k, T in enumerate(C13_TEMPS):\n            for s in range(SAMPLES):\n                g = torch.Generator(device=device).manual_seed(gen_seed13(k, s, arm))\n                rng = np.random.default_rng(coin_seed13(k, s, arm))\n                r = gen_chunked(model, prompts, T, valid, g, H2, kind, dose, rng, chunk)\n                for kk in keys_f + keys_b:\n                    G[kk][k, s] = r[kk]\n                toks[k, s] = r[\'toks\']\n        path = f\'{outdir}/{name}@{arm}.npz\'\n        np.savez_compressed(path + \'.part.npz\', toks=toks, temps=np.array(C13_TEMPS), use=np.array(use), eos=np.array(eos),\n                            prompts=np.array(prompts, np.int32), arm=arm, kind=kind, dose=dose, chunk=chunk or 0,\n                            seconds=time.time() - ta, **{f\'g_{k}\': v for k, v in G.items()})\n        os.replace(path + \'.part.npz\', path)\n        notify(f\'{name}@{arm} おわり（{time.time() - ta:.0f} 秒、通算 {time.time() - t0:.0f} 秒）\')\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\ndef self_test_13(device):\n    """設定の確かめと、分けて生成したものが決まった結果になり、禁止の割合が 30% に近いこと（小さなランダムのモデル）。"""\n    assert len(C13_TEMPS) == 7 and C13_TEMPS[0] == 0.4 and C13_TEMPS[-1] == 0.7\n    assert [a[0] for a in C13_ARMS] == [\'none\', \'none_b\', \'pl300\', \'pl300_b\'] and C13_SEED == 20261123\n    arms = [a[0] for a in C13_ARMS]\n    assert len({gen_seed13(k, s, a) for k in range(7) for s in range(2) for a in arms}) == 28        # なし と 偽薬 は同じ種（A2-10 と同じ）、_b は別\n    assert len({coin_seed13(k, s, a) for k in range(7) for s in range(2) for a in arms}) == 28\n    assert gen_seed13(0, 0, \'none_b\') != gen_seed13(0, 0, \'none\') and gen_seed13(0, 0, \'pl300_b\') != gen_seed13(0, 0, \'pl300\')\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global CONT\n    keep = CONT\n    CONT = 100\n    try:\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4, [20, 21, 22, 23] * 4] * 4\n        # 分けたときと分けないときは、multinomial が束ごとに乱数を引くので同じトークンにはならない。ここでは「分けたもの」が\n        # 決まった結果になること（2 回で同じ）と、形と、禁止の割合を確かめる。\n        a = gen_chunked(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(3), 33, \'placebo\', 0.3,\n                        np.random.default_rng(0), 8)\n        b = gen_chunked(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(3), 33, \'placebo\', 0.3,\n                        np.random.default_rng(0), 8)\n        assert (a[\'toks\'] == b[\'toks\']).all() and a[\'toks\'].shape == (16, 100)\n        assert abs(a[\'banned\'].mean() - 0.3) < 0.05, a[\'banned\'].mean()\n    finally:\n        CONT = keep\n    return \'OK\'\n'
assert hashlib.sha256(SRC13C.encode()).hexdigest() == 'fb19164812078c8ba1377f2245f431d8a73ae5e9b8fbab0bd124b05b9f884634'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRC13C)
C13_GROUP = 'small'
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ モデルごとに 4 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_13('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_13', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    for name, hf_id, chunk in C13_GROUPS[C13_GROUP]:
        run_13(name, hf_id, chunk, docs, '/content/drive/MyDrive/logit-thermo/A2_13', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(gate='docs/A2_13_GATE.md', group=C13_GROUP, models=C13_GROUPS[C13_GROUP], temps=C13_TEMPS, arms=[a[0] for a in C13_ARMS],
                   seed=C13_SEED, a2_7b_collect_sha256='b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', a2_13_collect_sha256='fb19164812078c8ba1377f2245f431d8a73ae5e9b8fbab0bd124b05b9f884634', torch=torch.__version__,
                   transformers=transformers.__version__, python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0),
                   matmul=torch.get_float32_matmul_precision()), open('/content/drive/MyDrive/logit-thermo/A2_13' + f'/run_meta_{C13_GROUP}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-13 のデータ集め（small）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-13 の small（Pythia-70M・160M・410M・1B、SmolLM2-135M・360M）を A2_13 に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
